In [ ]:
#| default_exp setup

# Launcher setup

> Create a "CDP Chrome" launcher for a debug-enabled, automation-only Chrome

The `connect` and extension transports drive your everyday browser, but sometimes that's exactly what you don't want: an agent filling forms shouldn't see your banking session. `fastcdp-setup` creates an OS launcher ("CDP Chrome") that starts your installed Chrome with `--remote-debugging-port` and its own profile directory, so you get a separate Chrome that's only logged in to the things you want automated, ready for `CDP.remote()`. On macOS it's a real app bundle in `~/Applications`, on Linux a `.desktop` entry, on Windows a Start Menu shortcut.

In [ ]:
#| export
from fastcore.utils import *
from fastcore.script import call_parse
import platform, subprocess, shutil
from importlib.resources import files

In [ ]:
from fastcore.test import test_eq

In [ ]:
#| export
def chrome_path():
    "Find the installed Chrome (or Chromium) executable"
    sys = platform.system()
    if sys == 'Darwin': return Path('/Applications/Google Chrome.app/Contents/MacOS/Google Chrome')
    if sys == 'Windows':
        cands = [Path(os.environ.get(v, ''))/'Google/Chrome/Application/chrome.exe'
                 for v in ('PROGRAMFILES', 'PROGRAMFILES(X86)', 'LOCALAPPDATA')]
        if (p := first(cands, Path.is_file)): return p
        raise FileNotFoundError('chrome.exe not found')
    for n in ('google-chrome', 'google-chrome-stable', 'chromium', 'chromium-browser'):
        if (p := shutil.which(n)): return Path(p)
    raise FileNotFoundError('chrome not found on PATH')

Each platform builder takes explicit `chrome`, `port`, `profile`, and `dest` paths, so they can be exercised anywhere; `main` picks the right one and the conventional destination for the OS.

In [ ]:
#| export
_PLIST = """<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE plist PUBLIC "-//Apple//DTD PLIST 1.0//EN" "http://www.apple.com/DTDs/PropertyList-1.0.dtd">
<plist version="1.0"><dict>
  <key>CFBundleName</key><string>{name}</string>
  <key>CFBundleDisplayName</key><string>{name}</string>
  <key>CFBundleIdentifier</key><string>ai.answer.fastcdp.{ident}</string>
  <key>CFBundleExecutable</key><string>launch</string>
  <key>CFBundleIconFile</key><string>icon</string>
  <key>CFBundlePackageType</key><string>APPL</string>
  <key>CFBundleVersion</key><string>1.0</string>
</dict></plist>
"""

def _ident(name): return re.sub(r'\W+', '-', name.lower()).strip('-')

def _mac_app(chrome, name, port, profile, dest):
    app = dest/f'{name}.app'
    (app/'Contents/MacOS').mkdir(parents=True, exist_ok=True)
    (app/'Contents/Resources').mkdir(parents=True, exist_ok=True)
    (app/'Contents/Info.plist').write_text(_PLIST.format(name=name, ident=_ident(name)))
    scr = app/'Contents/MacOS/launch'
    # `arch -arm64`: a script-bundle executable runs under Rosetta, and Chrome would inherit its x86_64 slice
    scr.write_text(f'#!/bin/sh\nexec arch -arm64 "{chrome}" --remote-debugging-port={port} --user-data-dir="{profile}" --no-first-run --no-default-browser-check "$@"\n')
    scr.chmod(0o755)
    shutil.copy(files('fastcdp')/'icons/cdp-chrome.icns', app/'Contents/Resources/icon.icns')
    return app

The bundle is three files: a plist, a one-line launcher script, and the icon. Building one into a temp dir shows the structure:

In [ ]:
app = _mac_app(Path('/usr/bin/true'), 'CDP Chrome', 9223, Path('/tmp/prof'), Path('/tmp/fastcdp-test-apps'))
print((app/'Contents/MacOS/launch').read_text())
test_eq((app/'Contents/Resources/icon.icns').exists(), True)

In [ ]:
#| export
def _linux_app(chrome, name, port, profile, dest):
    ic = dest/'icons/hicolor/256x256/apps/cdp-chrome.png'
    ic.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(files('fastcdp')/'icons/cdp-chrome.png', ic)
    desk = dest/'applications'/f'{_ident(name)}.desktop'
    desk.parent.mkdir(parents=True, exist_ok=True)
    desk.write_text(f'''[Desktop Entry]
Type=Application
Name={name}
Exec="{chrome}" --remote-debugging-port={port} --user-data-dir="{profile}" --no-first-run --no-default-browser-check
Icon=cdp-chrome
Terminal=false
Categories=Development;
''')
    return desk

A `.desktop` entry plus a `hicolor` icon is the whole story on Linux:

In [ ]:
desk = _linux_app(Path('/usr/bin/true'), 'CDP Chrome', 9223, Path('/tmp/prof'), Path('/tmp/fastcdp-test-share'))
print(desk.read_text())
test_eq(desk.name, 'cdp-chrome.desktop')

In [ ]:
#| export
def _win_app(chrome, name, port, profile, dest):
    ic = Path(profile)/'cdp-chrome.ico'
    ic.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(files('fastcdp')/'icons/cdp-chrome.ico', ic)
    dest.mkdir(parents=True, exist_ok=True)
    lnk = dest/f'{name}.lnk'
    ps = f'''$s = (New-Object -ComObject WScript.Shell).CreateShortcut('{lnk}')
$s.TargetPath = '{chrome}'
$s.Arguments = '--remote-debugging-port={port} --user-data-dir="{profile}" --no-first-run --no-default-browser-check'
$s.IconLocation = '{ic}'
$s.Save()'''
    subprocess.run(['powershell', '-NoProfile', '-Command', ps], check=True)
    return lnk

Windows gets a Start Menu shortcut built by PowerShell's `WScript.Shell` COM object; the icon is copied next to the profile so the `.lnk` has a stable path to point at.

In [ ]:
#| export
@call_parse
def main(
    port:int=9223, # Debug port (`CDP.remote`'s default; 9222 would clash with the everyday browser's built-in debugging)
    profile:str=None, # Chrome profile dir (default: `~/.cache/fastcdp/cdp-chrome`)
    name:str='CDP Chrome' # Launcher name
):
    "Create an OS launcher for a debug-enabled Chrome with its own profile, ready for `CDP.remote`"
    profile = Path(profile) if profile else Path.home()/'.cache/fastcdp/cdp-chrome'
    chrome, sys = chrome_path(), platform.system()
    if sys == 'Darwin': res = _mac_app(chrome, name, port, profile, Path.home()/'Applications')
    elif sys == 'Linux': res = _linux_app(chrome, name, port, profile, Path.home()/'.local/share')
    elif sys == 'Windows': res = _win_app(chrome, name, port, profile, Path(os.environ['APPDATA'])/'Microsoft/Windows/Start Menu/Programs')
    else: raise RuntimeError(f'Unsupported OS: {sys}')
    print(f'Created {res}\nConnect with: cdp = await CDP.remote({port})')
    return res

`fastcdp-setup` is registered as a console script, so after `pip install fastcdp` it's one command with sensible defaults. It shares `CDP.remote`'s default port 9223, so the pair works with no arguments on either side; 9222 is avoided because a main browser with built-in remote debugging enabled already holds it.